# 데이터 확인 및 전처리

## 1. 기본설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import shutil # 파일을 복사하기 위한 표준 라이브럴기
from pathlib import Path

import pandas as pd
from IPython.display import display


# 분석할 데이터셋
RATIOS = ["HI", "LI"]
SIZE = "Small"

SAMPLE_ROWS = 10_000

# Kaggle 데이터셋 주소
DATASET_HANDLE = (
    "ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml"
)

# 현재 노트북 위치에 datasets 폴더 생성
DATA_DIR = Path.cwd() / "drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets"
DATA_DIR.mkdir(exist_ok=True)

print("현재 작업 폴더:", Path.cwd())
print("데이터 저장 폴더:", DATA_DIR)
print("분석 대상:", RATIOS, SIZE)

현재 작업 폴더: /content
데이터 저장 폴더: /content/drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets
분석 대상: ['HI', 'LI'] Small


## 2. 다운로드할 파일 목록 만들기

In [5]:
import kagglehub

wanted_files = []

for ratio in RATIOS:
    wanted_files.extend([

        # 전체 거래 데이터
        f"{ratio}-{SIZE}_Trans.csv",

        # 계좌와 엔티티 정보
        f"{ratio}-{SIZE}_accounts.csv",

        # 자금세탁 패턴별 거래 목록
        f"{ratio}-{SIZE}_Patterns.txt"
    ])

print('다운로드 대상 파일:')

for filename in wanted_files:
    print("-", filename)


다운로드 대상 파일:
- HI-Small_Trans.csv
- HI-Small_accounts.csv
- HI-Small_Patterns.txt
- LI-Small_Trans.csv
- LI-Small_accounts.csv
- LI-Small_Patterns.txt


## 3. HI_Samll과 LI-Small 파일 다운로드

In [7]:
# 다운로드할 파일을 하나씩 확인
for filename in wanted_files:

    # 최종적으로 파일을 저장할 위치
    destination = DATA_DIR / filename

    # 해당 파일이 이미 datasets 폴더에 있으면 다시 받지 않음
    if destination.exists():
        print("이미 존재하므로 건너뜀:", destination.name)
        continue

    print("다운로드 시작:", filename)

    # Kaggle 캐시 폴더에 파일 하나만 다운로드
    # path=filename을 지정했기 때문에 전체 데이터셋을 받지 않음
    downloaded_path = Path(
        kagglehub.dataset_download(
            DATASET_HANDLE,
            path=filename,
        )
    )

    # Kaggle 캐시에 저장된 파일을 현재 datasets 폴더로 복사
    shutil.copy2(
        downloaded_path,
        destination,
    )

    print("다운로드 완료:", destination)

이미 존재하므로 건너뜀: HI-Small_Trans.csv
이미 존재하므로 건너뜀: HI-Small_accounts.csv
이미 존재하므로 건너뜀: HI-Small_Patterns.txt
이미 존재하므로 건너뜀: LI-Small_Trans.csv
이미 존재하므로 건너뜀: LI-Small_accounts.csv
이미 존재하므로 건너뜀: LI-Small_Patterns.txt


## 4. 다운로드된 파일 확인

In [8]:
# datasets 폴더 안의 파일만 가져와 파일명순으로 정렬
downloaded_files = sorted(
    path
    for path in DATA_DIR.iterdir()
    if path.is_file()
)

print("현재 datasets 폴더의 파일 목록")
print("=" * 70)

for path in downloaded_files:
    # 파일 크기를 byte에서 MB로 변환
    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{path.name:<35}"
        f"{size_mb:>12.2f} MB"
    )

현재 datasets 폴더의 파일 목록
HI-Small_Patterns.txt                      0.31 MB
HI-Small_Trans.csv                       453.63 MB
HI-Small_accounts.csv                     32.48 MB
LI-Small_Patterns.txt                      0.09 MB
LI-Small_Trans.csv                       620.29 MB
LI-Small_accounts.csv                     45.06 MB


## 5. HI와 LI 파일 경로를 딕셔너리로 정리

In [9]:
# 데이터셋별 파일 경로를 저장할 딕셔너리
dataset_paths = {}

for ratio in RATIOS:
    dataset_paths[ratio] = {
        "transactions": (
            DATA_DIR / f"{ratio}-{SIZE}_Trans.csv"
        ),
        "accounts": (
            DATA_DIR / f"{ratio}-{SIZE}_accounts.csv"
        ),
        "patterns": (
            DATA_DIR / f"{ratio}-{SIZE}_Patterns.txt"
        ),
    }


# 모든 파일이 실제로 존재하는지 확인
for ratio, paths in dataset_paths.items():
    print(f"\n===== {ratio}-{SIZE} =====")

    for file_type, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(
                f"파일이 존재하지 않습니다: {path}"
            )

        print(f"{file_type:<15}: {path.name}")


===== HI-Small =====
transactions   : HI-Small_Trans.csv
accounts       : HI-Small_accounts.csv
patterns       : HI-Small_Patterns.txt

===== LI-Small =====
transactions   : LI-Small_Trans.csv
accounts       : LI-Small_accounts.csv
patterns       : LI-Small_Patterns.txt


In [10]:
dataset_paths

{'HI': {'transactions': PosixPath('/content/drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets/HI-Small_Trans.csv'),
  'accounts': PosixPath('/content/drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets/HI-Small_accounts.csv'),
  'patterns': PosixPath('/content/drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets/HI-Small_Patterns.txt')},
 'LI': {'transactions': PosixPath('/content/drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets/LI-Small_Trans.csv'),
  'accounts': PosixPath('/content/drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets/LI-Small_accounts.csv'),
  'patterns': PosixPath('/content/drive/MyDrive/Colab Notebooks/인공지능사관학교/기업-17/datasets/LI-Small_Patterns.txt')}}

## 6. 데이터 로드
- 일부만 할거면 주석 풀기

In [33]:
transaction_dtypes = {
    "From Bank": "string",
    "Account": "string",
    "To Bank": "string",
    "Account.1": "string",
}

account_dtypes = {
    "Bank ID": "string",
    "Account Number": "string",
    "Entity ID": "string",
}

raw_samples = {}

for ratio in RATIOS:
    print(f"{ratio}-{SIZE} 샘플 데이터 로드 중.. ")

    transactions = pd.read_csv(
        dataset_paths[ratio]['transactions'],
        dtype=transaction_dtypes, # 데이터 타입 str로 맞추기
        # nrows = SAMPLE_ROWS,
        parse_dates=["Timestamp"],
        low_memory = False,
    )

    accounts = pd.read_csv(
        dataset_paths[ratio]['accounts'],
        dtype=account_dtypes,
        # nrows = SAMPLE_ROWS,
        low_memory = False,
    )

    raw_samples[ratio] = {
        "transactions" : transactions,
        "accounts" : accounts,
    }

    print(
        f"{ratio} 거래 데이터크기",
        transactions.shape
    )


    print(
        f"{ratio} 계좌 데이터크기",
        accounts.shape
    )



HI-Small 샘플 데이터 로드 중.. 
HI 거래 데이터크기 (5078345, 11)
HI 계좌 데이터크기 (518581, 5)
LI-Small 샘플 데이터 로드 중.. 
LI 거래 데이터크기 (6924049, 11)
LI 계좌 데이터크기 (712688, 5)


## 7. 거래 데이터 확인

In [34]:
hi_trans_raw = raw_samples['HI']["transactions"]

print("===== HI-Small 거래 데이터 =====")
print("현재 불러온 크기:", hi_trans_raw.shape)

hi_trans_raw.info(
    verbose = True,
    show_counts = True,
    memory_usage = "deep",
)

===== HI-Small 거래 데이터 =====
현재 불러온 크기: (5078345, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   Timestamp           5078345 non-null  datetime64[ns]
 1   From Bank           5078345 non-null  string        
 2   Account             5078345 non-null  string        
 3   To Bank             5078345 non-null  string        
 4   Account.1           5078345 non-null  string        
 5   Amount Received     5078345 non-null  float64       
 6   Receiving Currency  5078345 non-null  object        
 7   Amount Paid         5078345 non-null  float64       
 8   Payment Currency    5078345 non-null  object        
 9   Payment Format      5078345 non-null  object        
 10  Is Laundering       5078345 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(3), string(4)
memory usa

In [35]:
hi_trans_raw.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,010,8000EBD30,010,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,03208,8000F4580,001,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,012,8000F5030,012,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,010,8000F5200,010,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [36]:
li_trans_raw = raw_samples['LI']["transactions"]

print("===== LI-Small 거래 데이터 =====")
print("현재 불러온 크기:", li_trans_raw.shape)

li_trans_raw.info(
    verbose = True,
    show_counts = True,
    memory_usage = "deep",
)

===== LI-Small 거래 데이터 =====
현재 불러온 크기: (6924049, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   Timestamp           6924049 non-null  datetime64[ns]
 1   From Bank           6924049 non-null  string        
 2   Account             6924049 non-null  string        
 3   To Bank             6924049 non-null  string        
 4   Account.1           6924049 non-null  string        
 5   Amount Received     6924049 non-null  float64       
 6   Receiving Currency  6924049 non-null  object        
 7   Amount Paid         6924049 non-null  float64       
 8   Payment Currency    6924049 non-null  object        
 9   Payment Format      6924049 non-null  object        
 10  Is Laundering       6924049 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(3), string(4)
memory usa

In [37]:
li_trans_raw.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:08:00,011,8000ECA90,011,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022-09-01 00:21:00,03402,80021DAD0,03402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022-09-01 00:00:00,011,8000ECA90,001120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022-09-01 00:16:00,03814,8006AD080,03814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022-09-01 00:00:00,020,8006AD530,020,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


## 8. 계좌 데이터 확이

In [38]:
hi_accounts_raw = raw_samples['HI']['accounts']

print("===== HI-Small 계좌 데이터")
print("현재 불러온 크기: ", hi_accounts_raw.shape)

hi_accounts_raw.info()

===== HI-Small 계좌 데이터
현재 불러온 크기:  (518581, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 518581 entries, 0 to 518580
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   Bank Name       518581 non-null  object
 1   Bank ID         518581 non-null  string
 2   Account Number  518581 non-null  string
 3   Entity ID       518581 non-null  string
 4   Entity Name     518581 non-null  object
dtypes: object(2), string(3)
memory usage: 19.8+ MB


In [39]:
li_accounts_raw = raw_samples['LI']['accounts']

print("===== LI-Small 계좌 데이터")
print("현재 불러온 크기: ", li_accounts_raw.shape)

li_accounts_raw.info()

===== LI-Small 계좌 데이터
현재 불러온 크기:  (712688, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712688 entries, 0 to 712687
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   Bank Name       712688 non-null  object
 1   Bank ID         712688 non-null  string
 2   Account Number  712688 non-null  string
 3   Entity ID       712688 non-null  string
 4   Entity Name     712688 non-null  object
dtypes: object(2), string(3)
memory usage: 27.2+ MB


## 9. 컬럼별 원본 상태 확인

In [40]:
def inspect_raw_dataframe(df, dataset_name):
    """
    DataFrame의 원본 상태를 확인하는 함수.

    확인하는 내용
    -------------
    1. 행과 열의 개수
    2. Pandas가 자동으로 판단한 dtype
    3. 결측값 개수와 비율
    4. 현재 샘플 안에서의 고유값 개수
    5. 각 컬럼의 실제 값 예시

    주의
    ----
    이 함수는 DataFrame을 변경하지 않는다.
    """

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    print("행과 열:", df.shape)

    print("\n[DataFrame info]")

    df.info()

    # 컴럼별 검사 결과를 저장할 리스트
    profile_rows = []

    for column in df.columns:
        series = df[column]

        # 결측값을 제외한 값
        non_null = series.dropna()

        # 중복되지 않은 실제 값 예시 최대 3개
        examples = (
            non_null
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        profile_rows.append({
            "column": column,

            # Pandas가 자동으로 판단한 자료형
            "pandas_dtype": str(series.dtype),

            # 결측값 개수
            "missing_count": int(
                series.isna().sum()
            ),

            # 결측값 비율
            "missing_ratio_percent": round(
                series.isna().mean() * 100,
                4,
            ),

            # 현재 읽은 샘플 안에서의 고유값 수
            "unique_count_in_sample": int(
                series.nunique(dropna=True)
            ),

            # 실제 값 예시
            "examples": examples,

        })

    # 검사 결과를 표로 변환
    profile = pd.DataFrame(profile_rows)

    print("\n[컬럼별 프로파일]")
    display(profile)

    print("\n[앞의 10행]")
    display(df.head(10))

    return profile

In [41]:
transactions_profiles = {}

for ratio in RATIOS:
    transactions_profiles[ratio] = inspect_raw_dataframe(
        raw_samples[ratio]["transactions"],
        f"{ratio}-{SIZE} 거래 데이터"
    )



HI-Small 거래 데이터
행과 열: (5078345, 11)

[DataFrame info]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   Timestamp           datetime64[ns]
 1   From Bank           string        
 2   Account             string        
 3   To Bank             string        
 4   Account.1           string        
 5   Amount Received     float64       
 6   Receiving Currency  object        
 7   Amount Paid         float64       
 8   Payment Currency    object        
 9   Payment Format      object        
 10  Is Laundering       int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(3), string(4)
memory usage: 426.2+ MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Timestamp,datetime64[ns],0,0.0,15018,"[2022-09-01 00:20:00, 2022-09-01 00:00:00, 202..."
1,From Bank,string,0,0.0,30528,"[010, 03208, 03209]"
2,Account,string,0,0.0,496995,"[8000EBD30, 8000F4580, 8000F4670]"
3,To Bank,string,0,0.0,15850,"[010, 001, 03209]"
4,Account.1,string,0,0.0,420636,"[8000EBD30, 8000F5340, 8000F4670]"
5,Amount Received,float64,0,0.0,915161,"[3697.34, 0.01, 14675.57]"
6,Receiving Currency,object,0,0.0,15,"[US Dollar, Bitcoin, Euro]"
7,Amount Paid,float64,0,0.0,923873,"[3697.34, 0.01, 14675.57]"
8,Payment Currency,object,0,0.0,15,"[US Dollar, Bitcoin, Euro]"
9,Payment Format,object,0,0.0,7,"[Reinvestment, Cheque, Credit Card]"



[앞의 10행]


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,010,8000EBD30,010,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,03208,8000F4580,001,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,012,8000F5030,012,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,010,8000F5200,010,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0
5,2022-09-01 00:03:00,001,8000F5AD0,001,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0
6,2022-09-01 00:08:00,001,8000EBAC0,001,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0
7,2022-09-01 00:16:00,001,8000EC1E0,001,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0
8,2022-09-01 00:26:00,012,8000EC280,002439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0
9,2022-09-01 00:21:00,001,8000EDEC0,0211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0



LI-Small 거래 데이터
행과 열: (6924049, 11)

[DataFrame info]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   Timestamp           datetime64[ns]
 1   From Bank           string        
 2   Account             string        
 3   To Bank             string        
 4   Account.1           string        
 5   Amount Received     float64       
 6   Receiving Currency  object        
 7   Amount Paid         float64       
 8   Payment Currency    object        
 9   Payment Format      object        
 10  Is Laundering       int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(3), string(4)
memory usage: 581.1+ MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Timestamp,datetime64[ns],0,0.0,14533,"[2022-09-01 00:08:00, 2022-09-01 00:21:00, 202..."
1,From Bank,string,0,0.0,41866,"[011, 03402, 03814]"
2,Account,string,0,0.0,681281,"[8000ECA90, 80021DAD0, 8006AD080]"
3,To Bank,string,0,0.0,21620,"[011, 03402, 001120]"
4,Account.1,string,0,0.0,576176,"[8000ECA90, 80021DAD0, 8006AA910]"
5,Amount Received,float64,0,0.0,1194921,"[3195403.0, 1858.96, 592571.0]"
6,Receiving Currency,object,0,0.0,15,"[US Dollar, Euro, Bitcoin]"
7,Amount Paid,float64,0,0.0,1204309,"[3195403.0, 1858.96, 592571.0]"
8,Payment Currency,object,0,0.0,15,"[US Dollar, Euro, Bitcoin]"
9,Payment Format,object,0,0.0,7,"[Reinvestment, Cheque, ACH]"



[앞의 10행]


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:08:00,011,8000ECA90,011,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022-09-01 00:21:00,03402,80021DAD0,03402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022-09-01 00:00:00,011,8000ECA90,001120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022-09-01 00:16:00,03814,8006AD080,03814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022-09-01 00:00:00,020,8006AD530,020,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0
5,2022-09-01 00:24:00,012,8006ADD30,012,8006ADD30,6473.62,US Dollar,6473.62,US Dollar,Reinvestment,0
6,2022-09-01 00:17:00,011,800059120,01217,8006AD4E0,60562.00,US Dollar,60562.00,US Dollar,ACH,0
7,2022-09-01 00:07:00,011,8000ECA90,011,8000ECA90,22.97,US Dollar,22.97,US Dollar,Reinvestment,0
8,2022-09-01 00:28:00,001120,8006AA910,0243166,81470DCF0,43.53,US Dollar,43.53,US Dollar,Credit Card,0
9,2022-09-01 00:22:00,01217,8006AD4E0,01217,8006AD4E0,5.04,US Dollar,5.04,US Dollar,Reinvestment,0


In [42]:
account_profiles = {}

for ratio in RATIOS:
    account_profiles[ratio] = inspect_raw_dataframe(
        raw_samples[ratio]["accounts"],
        f"{ratio}-{SIZE} 계좌 데이터 "
    )


HI-Small 계좌 데이터 
행과 열: (518581, 5)

[DataFrame info]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 518581 entries, 0 to 518580
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   Bank Name       518581 non-null  object
 1   Bank ID         518581 non-null  string
 2   Account Number  518581 non-null  string
 3   Entity ID       518581 non-null  string
 4   Entity Name     518581 non-null  object
dtypes: object(2), string(3)
memory usage: 19.8+ MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Bank Name,object,0,0.0,20053,"[Portugal Bank #4507, Canada Bank #27, UK Bank..."
1,Bank ID,string,0,0.0,30470,"[331579, 210, 21884]"
2,Account Number,string,0,0.0,518573,"[80B779D80, 809D86900, 80812BE00]"
3,Entity ID,string,0,0.0,166207,"[80062E240, 800C998A0, 800C47F50]"
4,Entity Name,object,0,0.0,166207,"[Sole Proprietorship #50438, Corporation #3352..."



[앞의 10행]


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889
5,Spain Bank #439,224555,80F269580,80064EB20,Sole Proprietorship #42987
6,Savings Bank of Omaha,32013,80E6E8680,800CCEDE0,Sole Proprietorship #20855
7,Brazil Bank #39,335355,80CDEFD80,800D88F10,Partnership #36822
8,Mexico Bank #16,1132,80B723600,800D3F760,Partnership #36511
9,Russia Bank #39,217824,806B17000,800B44480,Partnership #33830



LI-Small 계좌 데이터 
행과 열: (712688, 5)

[DataFrame info]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712688 entries, 0 to 712687
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   Bank Name       712688 non-null  object
 1   Bank ID         712688 non-null  string
 2   Account Number  712688 non-null  string
 3   Entity ID       712688 non-null  string
 4   Entity Name     712688 non-null  object
dtypes: object(2), string(3)
memory usage: 27.2+ MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Bank Name,object,0,0.0,27652,"[China Bank #2820, France Bank #4585, China Ba..."
1,Bank ID,string,0,0.0,41815,"[314693, 311253, 39996]"
2,Account Number,string,0,0.0,712684,"[81B86A280, 8187FEA80, 803961E00]"
3,Entity ID,string,0,0.0,224931,"[800D8CCF0, 800B505E0, 800D03F60]"
4,Entity Name,object,0,0.0,224931,"[Corporation #41344, Corporation #54497, Partn..."



[앞의 10행]


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930
5,Savings Bank of Harrisburg,270107,81A2F8D80,80145CEC0,Partnership #17131
6,Crytpo Bank #22,170412,819F4D601,8015B10D0,Partnership #953
7,Brazil Bank #19,148894,811EACF00,801208FE0,Corporation #48389
8,Japan Bank #654,322173,808753E80,800E622A0,Corporation #42450
9,Italy Bank #1554,52130,81455D100,800C1C190,Sole Proprietorship #29256


## 10. 범주형 컬럼 값 확인

In [43]:
# 거래 데이터에서 값의 종류를 확인할 컬럼
columns_to_check = [
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

for ratio in RATIOS:
    df = raw_samples[ratio]["transactions"]

    print("\n" + "=" * 80)
    print(f"{ratio}-SIZE 주요 범주형 컬럼")
    print("=" * 80)

    for column in columns_to_check:
        # 해당 컬럼이 없으면 건너뜀
        if column not in df.columns:
            print("컬럼 없음:", column)
            continue

        print(f"\n[{column}]")

        # 결속값도 포함에 값별 개수 계산
        counts = (
            df[column]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

        counts['ratio_percent'] = (
            counts['count']
            / len(df)
            * 100
        ).round(4)

        display(counts.head(30))


HI-SIZE 주요 범주형 컬럼

[Receiving Currency]


,count,ratio_percent
Receiving Currency,,
US Dollar,1879341,37.0070
Euro,1172017,23.0787
Swiss Franc,237884,4.6843
Yuan,206551,4.0673
Shekel,194988,3.8396
Rupee,192065,3.7820
UK Pound,181255,3.5692
Ruble,157361,3.0987
Yen,156319,3.0781



[Payment Currency]


,count,ratio_percent
Payment Currency,,
US Dollar,1895172,37.3187
Euro,1168297,23.0055
Swiss Franc,234860,4.6247
Yuan,213752,4.2091
Shekel,192184,3.7844
Rupee,190202,3.7454
UK Pound,180738,3.5590
Yen,155209,3.0563
Ruble,155178,3.0557



[Payment Format]


...

,count,ratio_percent
Payment Currency,,
US Dollar,2553887,36.8843
Euro,1595859,23.0481
Yuan,483603,6.9844
Rupee,340641,4.9197
Bitcoin,309240,4.4662
Saudi Riyal,257948,3.7254
Australian Dollar,211155,3.0496
Yen,210125,3.0347
Brazil Real,199840,2.8862



[Payment Format]


,count,ratio_percent
Payment Format,,
Cheque,2503158,36.1517
Credit Card,1780389,25.7131
ACH,796581,11.5046
Cash,655688,9.4697
Reinvestment,650458,9.3942
Bitcoin,309208,4.4657
Wire,228567,3.3011



[Is Laundering]


,count,ratio_percent
Is Laundering,,
0,6920484,99.9485
1,3565,0.0515


## 11. 수치형 컬럼값 확인

In [44]:
for ratio in RATIOS:
    df = raw_samples[ratio]['transactions']

    print("\n" + "=" * 80)
    print(f"{ratio}-{SIZE} 숫자형 컬럼 요약")
    print("=" * 80)

    numeric_summary = df.describe(
        include="number",
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ],
    ).T

    display(numeric_summary)



HI-Small 숫자형 컬럼 요약


,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
Amount Received,5078345.0,5.988726e+06,1.037183e+09,0.000001,0.016808,183.37,1411.01,12346.27,649906.51,1.475123e+07,1.046302e+12
Amount Paid,5078345.0,4.509273e+06,8.697728e+08,0.000001,0.018477,184.48,1414.54,12297.84,623757.22,1.352453e+07,1.046302e+12
Is Laundering,5078345.0,1.019427e-03,3.191219e-02,0.000000,0.000000,0.00,0.00,0.00,0.00,0.000000e+00,1.000000e+00



LI-Small 숫자형 컬럼 요약


,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
Amount Received,6924049.0,6.324067e+06,2.105371e+09,0.000001,0.009488,174.21,1397.62,12296.33,623594.670,1.336849e+07,3.644854e+12
Amount Paid,6924049.0,4.676036e+06,1.544099e+09,0.000001,0.010000,175.38,1399.44,12226.87,600085.324,1.236483e+07,3.644854e+12
Is Laundering,6924049.0,5.148722e-04,2.268495e-02,0.000000,0.000000,0.00,0.00,0.00,0.000,0.000000e+00,1.000000e+00


## 12. 문자열 원본과 Pandas 자동 추론 결과 비교

In [45]:
text_samples = {}

for ratio in RATIOS:
    # 모든 컬럼을 문자열로 읽음
    text_samples[ratio] = pd.read_csv(
        dataset_paths[ratio]["transactions"],
        nrows=100,
        dtype="string",
        low_memory=False,
    )

In [46]:
id_columns = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
]

for ratio in RATIOS:
    default_df = raw_samples[ratio]["transactions"]
    text_df = text_samples[ratio]

    print("\n" + "=" * 80)
    print(f"{ratio}-{SIZE} ID 컬럼 비교")
    print("=" * 80)

    for column in id_columns:
        if column not in default_df.columns:
            continue

        print(f"\n[{column}]")
        print(
            "Pandas 자동 dtype:",
            default_df[column].dtype,
        )

        comparison = pd.DataFrame({
            # Pandas가 자동으로 추론한 결과
            "pandas_default": (
                default_df[column]
                .head(20)
                .reset_index(drop=True)
            ),

            # 원본을 문자열로 강제해서 읽은 결과
            "original_as_string": (
                text_df[column]
                .head(20)
                .reset_index(drop=True)
            ),
        })

        display(comparison)


HI-Small ID 컬럼 비교

[From Bank]
Pandas 자동 dtype: string


,pandas_default,original_as_string
0,010,010
1,03208,03208
2,03209,03209
3,012,012
4,010,010
5,001,001
6,001,001
7,001,001
8,012,012
9,001,001



[Account]
Pandas 자동 dtype: string


,pandas_default,original_as_string
0,8000EBD30,8000EBD30
1,8000F4580,8000F4580
2,8000F4670,8000F4670
3,8000F5030,8000F5030
4,8000F5200,8000F5200
5,8000F5AD0,8000F5AD0
6,8000EBAC0,8000EBAC0
7,8000EC1E0,8000EC1E0
8,8000EC280,8000EC280
9,8000EDEC0,8000EDEC0



[To Bank]
Pandas 자동 dtype: string


...

,pandas_default,original_as_string
0,8000ECA90,8000ECA90
1,80021DAD0,80021DAD0
2,8000ECA90,8000ECA90
3,8006AD080,8006AD080
4,8006AD530,8006AD530
5,8006ADD30,8006ADD30
6,800059120,800059120
7,8000ECA90,8000ECA90
8,8006AA910,8006AA910
9,8006AD4E0,8006AD4E0



[To Bank]
Pandas 자동 dtype: string


,pandas_default,original_as_string
0,011,011
1,03402,03402
2,001120,001120
3,03814,03814
4,020,020
5,012,012
6,01217,01217
7,011,011
8,0243166,0243166
9,01217,01217



[Account.1]
Pandas 자동 dtype: string


,pandas_default,original_as_string
0,8000ECA90,8000ECA90
1,80021DAD0,80021DAD0
2,8006AA910,8006AA910
3,8006AD080,8006AD080
4,8006AD530,8006AD530
5,8006ADD30,8006ADD30
6,8006AD4E0,8006AD4E0
7,8000ECA90,8000ECA90
8,81470DCF0,81470DCF0
9,8006AD4E0,8006AD4E0


## 13. HI 와 LI 패턴 파일 원문 확인

In [47]:
# 각 패턴 파일에서 확인할 최대 줄 수
PATTERN_PREVIEW_LINES = 50

for ratio in RATIOS:
    path = dataset_paths[ratio]["patterns"]

    print("\n" + "=" * 80)
    print(f"{ratio}-{SIZE} 패턴 파일 앞부분")
    print("=" * 80)

    with open(
        path,
        mode="r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):
            print(
                f"{line_number:>3}: "
                f"{line.rstrip()}"
            )

            if line_number >= PATTERN_PREVIEW_LINES:
                break


HI-Small 패턴 파일 앞부분
  1: BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
  2: 2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
  3: 2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
  4: 2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
  5: 2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
  6: 2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1
  7: 2022/09/01 13:13,021174,800737690,00513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1
  8: 2022/09/01 14:11,021174,800737690,020,80066B990,12536.92,Euro,12536.92,Euro,ACH,1
  9: 2022/09/02 15:40,021174,800737690,00410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1
 10: 2022/09/02 21:23,021174,800737690,01292,8004030A0,16135.09,US Dollar,16135.09,US Dollar,ACH,1
 11: 2022/09/02 23:10,021174,800737690,01601,800578800,12183.28,US Dollar,12183.28,US Do

## 14. 패턴 이름별 attempt 개수만 확인

In [48]:
from collections import Counter

pattern_attempt_counts = {}

for ratio in RATIOS:
    path = dataset_paths[ratio]["patterns"]

    # 패턴별 등장 횟수를 저장
    counter = Counter()

    with open(
        path,
        mode="r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line in file:
            line = line.strip()

            prefix = "BEGIN LAUNDERING ATTEMPT - "

            # 패턴 시작 행만 선택
            if not line.startswith(prefix):
                continue

            # 접두사 이후의 패턴 정보 추출
            pattern_meta = line[len(prefix):]

            # "CYCLE: Max 12 hops"처럼 부가 정보가 있다면
            # 콜론 앞부분만 패턴 이름으로 사용
            pattern_type = (
                pattern_meta
                .split(":", 1)[0]
                .strip()
                .upper()
            )

            counter[pattern_type] += 1

    pattern_attempt_counts[ratio] = counter

In [49]:
for ratio in RATIOS:
    print(f"\n===== {ratio}-{SIZE} 패턴별 attempt 수 =====")

    result = (
        pd.Series(
            pattern_attempt_counts[ratio],
            name="attempt_count",
        )
        .sort_index()
        .to_frame()
    )

    display(result)


===== HI-Small 패턴별 attempt 수 =====


,attempt_count
BIPARTITE,49
CYCLE,54
FAN-IN,40
FAN-OUT,48
GATHER-SCATTER,51
RANDOM,41
SCATTER-GATHER,44
STACK,43



===== LI-Small 패턴별 attempt 수 =====


,attempt_count
BIPARTITE,16
CYCLE,12
FAN-IN,12
FAN-OUT,19
GATHER-SCATTER,12
RANDOM,15
SCATTER-GATHER,13
STACK,18


### 결과정리
- Timestamp는 현재 문자열이므로 이후 datetime64[ns]로 변환해야 함
- From Bank, To Bank, Bank ID는 현재 int64임 - 식별자는 string으로 관리하는 것이 적절해 보임
- HI의 Account Number는 518,581행 중 518,573개로 계좌번호가 8개 중복
- LI의 Account Number는 712,688행 중 712,684개로 계좌번호가 4개 중복
- Entity ID와 Entity Name의 유니크 수는 두 데이터셋 모두 동일, Entity ID가 하나의 Entity Name에 대응하는 것으로 보이지만, 별도 교차표로 확인해야함


## 15. 중복된 모든 행 확인

In [50]:
duplicate_account_numbers = {}

for ratio in RATIOS:
    accounts = raw_samples[ratio]["accounts"]

    duplicated_rows = (
        accounts.loc[
            accounts["Account Number"].duplicated(keep=False)
        ]
        .sort_values(
            ["Account Number", "Bank ID"]
        )
        .reset_index(drop=True)
    )

    duplicate_account_numbers[ratio] = duplicated_rows

    print(
        f"\n===== {ratio}-{SIZE} "
        "중복 Account Number 전체 행 ====="
    )
    print("중복 관련 행 수:", len(duplicated_rows))
    print(
        "중복된 Account Number 종류 수:",
        duplicated_rows["Account Number"].nunique(),
    )

    display(duplicated_rows)


===== HI-Small 중복 Account Number 전체 행 =====
중복 관련 행 수: 16
중복된 Account Number 종류 수: 8


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Australia Bank #47,27755,80A7FD400,800CCD520,Corporation #33736
1,Australia Bank #44,28248,80A7FD400,800D28430,Partnership #36506
2,Australia Bank #47,27755,80A7FDE00,800D28430,Partnership #36506
3,Australia Bank #44,28248,80A7FDE00,800D15C50,Corporation #34165
4,Italy Bank #97,13858,80FA55EF0,8007F7670,Corporation #22542
5,Germany Bank #908,138832,80FA55EF0,8009067F0,Corporation #49502
6,Italy Bank #97,13858,80FA56340,80087BC70,Corporation #25811
7,Germany Bank #908,138832,80FA56340,8006CF910,Partnership #28810
8,Germany Bank #945,142574,81211BA20,8007D4530,Partnership #27156
9,Italy Bank #95,1490,81211BA20,80078A8D0,Corporation #24600



===== LI-Small 중복 Account Number 전체 행 =====
중복 관련 행 수: 8
중복된 Account Number 종류 수: 4


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Savings Bank of Columbus,113213,817037B20,80129EC80,Corporation #795
1,Savings Bank of the South,61144,817037B20,8007D0D30,Corporation #54169
2,Savings Bank of Columbus,113213,817038DF0,8014110E0,Corporation #11573
3,Savings Bank of the South,61144,817038DF0,80100DA30,Partnership #788
4,Spain Bank #507,144720,8177C8ED0,800C73BF0,Corporation #38102
5,Estonia Bank #2218,144840,8177C8ED0,800B01A00,Sole Proprietorship #31292
6,Spain Bank #507,144720,8177C94B0,800981A00,Partnership #32750
7,Estonia Bank #2218,144840,8177C94B0,800BB3280,Corporation #35681


In [51]:
for ratio in RATIOS:
    transactions = raw_samples[ratio]["transactions"]

    duplicate_mask = transactions.duplicated(
        keep=False
    )

    duplicate_rows = (
        transactions.loc[duplicate_mask]
        .sort_values(list(transactions.columns))
        .reset_index(drop=True)
    )

    print(f"\n===== {ratio}-{SIZE} 거래 중복 확인 =====")
    print(
        "중복을 제외한 추가 행 수:",
        transactions.duplicated().sum(),
    )
    print(
        "중복에 포함된 전체 행 수:",
        duplicate_mask.sum(),
    )

    display(duplicate_rows.head(100))


===== HI-Small 거래 중복 확인 =====
중복을 제외한 추가 행 수: 9
중복에 포함된 전체 행 수: 18


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 16:20:00,012004,800C927C1,012004,800C927C0,0.000008,Bitcoin,0.080000,Euro,ACH,0
1,2022-09-01 16:20:00,012004,800C927C1,012004,800C927C0,0.000008,Bitcoin,0.080000,Euro,ACH,0
2,2022-09-01 16:20:00,012004,800C927C1,0220,813D8C1E1,0.000008,Bitcoin,0.000008,Bitcoin,Bitcoin,0
3,2022-09-01 16:20:00,012004,800C927C1,0220,813D8C1E1,0.000008,Bitcoin,0.000008,Bitcoin,Bitcoin,0
4,2022-09-07 21:25:00,029992,8099A29B1,0220,813725AE1,0.000003,Bitcoin,0.000003,Bitcoin,Bitcoin,0
5,2022-09-07 21:25:00,029992,8099A29B1,0220,813725AE1,0.000003,Bitcoin,0.000003,Bitcoin,Bitcoin,0
6,2022-09-08 21:05:00,0113779,811144AB1,0053744,813C777F1,0.000002,Bitcoin,0.000002,Bitcoin,Bitcoin,0
7,2022-09-08 21:05:00,0113779,811144AB1,0053744,813C777F1,0.000002,Bitcoin,0.000002,Bitcoin,Bitcoin,0
8,2022-09-08 21:05:00,0113779,811144AB1,0113779,811144AB0,0.000002,Bitcoin,0.020000,US Dollar,ACH,0
9,2022-09-08 21:05:00,0113779,811144AB1,0113779,811144AB0,0.000002,Bitcoin,0.020000,US Dollar,ACH,0



===== LI-Small 거래 중복 확인 =====
중복을 제외한 추가 행 수: 8
중복에 포함된 전체 행 수: 16


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 05:03:00,021260,8085A09D1,0072146,81AB967E1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
1,2022-09-01 05:03:00,021260,8085A09D1,0072146,81AB967E1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
2,2022-09-01 07:44:00,0231616,814E45511,0171546,81B324AC1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
3,2022-09-01 07:44:00,0231616,814E45511,0171546,81B324AC1,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
4,2022-09-02 11:09:00,028335,804B468F1,0171669,81BBCDAB1,0.000007,Bitcoin,0.000007,Bitcoin,Bitcoin,0
5,2022-09-02 11:09:00,028335,804B468F1,0171669,81BBCDAB1,0.000007,Bitcoin,0.000007,Bitcoin,Bitcoin,0
6,2022-09-05 12:44:00,022661,80D016C81,0269294,81A245E81,0.000005,Bitcoin,0.000005,Bitcoin,Bitcoin,0
7,2022-09-05 12:44:00,022661,80D016C81,0269294,81A245E81,0.000005,Bitcoin,0.000005,Bitcoin,Bitcoin,0
8,2022-09-06 14:16:00,019970,804E44F11,0169489,8197DC971,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0
9,2022-09-06 14:16:00,019970,804E44F11,0169489,8197DC971,0.000001,Bitcoin,0.000001,Bitcoin,Bitcoin,0


## 16. 중복제거

In [52]:
processed_samples = {}

for ratio in RATIOS:
    transactions = (
        raw_samples[ratio]["transactions"]
        .drop_duplicates()
        .reset_index(drop=True)
        .copy()
    )

    accounts = (
        raw_samples[ratio]["accounts"]
        .copy()
    )

    processed_samples[ratio] = {
        "transactions": transactions,
        "accounts": accounts,
    }

    removed_count = (
        len(raw_samples[ratio]["transactions"])
        - len(transactions)
    )

    print(
        f"{ratio}-{SIZE}: "
        f"완전 중복 거래 {removed_count:,}행 제거"
    )

HI-Small: 완전 중복 거래 9행 제거
LI-Small: 완전 중복 거래 8행 제거


In [53]:
for ratio in RATIOS:
    transactions = processed_samples[ratio]["transactions"]

    print(
        ratio,
        "남은 완전 중복:",
        transactions.duplicated().sum(),
    )

HI 남은 완전 중복: 0
LI 남은 완전 중복: 0


## 17. 전치리된 컬럼 확인

In [55]:
for ratio in RATIOS:
    transactions = processed_samples[ratio]["transactions"]
    accounts = processed_samples[ratio]["accounts"]

    print(f"\n===== {ratio}-{SIZE} 거래 타입 =====")
    display(
        transactions[
            [
                "Timestamp",
                "From Bank",
                "Account",
                "To Bank",
                "Account.1",
            ]
        ].dtypes.to_frame("dtype")
    )

    print(f"\n===== {ratio}-{SIZE} 계좌 타입 =====")
    display(
        accounts[
            [
                "Bank ID",
                "Account Number",
                "Entity ID",
            ]
        ].dtypes.to_frame("dtype")
    )


===== HI-Small 거래 타입 =====


,dtype
Timestamp,datetime64[ns]
From Bank,string[python]
Account,string[python]
To Bank,string[python]
Account.1,string[python]



===== HI-Small 계좌 타입 =====


,dtype
Bank ID,string[python]
Account Number,string[python]
Entity ID,string[python]



===== LI-Small 거래 타입 =====


,dtype
Timestamp,datetime64[ns]
From Bank,string[python]
Account,string[python]
To Bank,string[python]
Account.1,string[python]



===== LI-Small 계좌 타입 =====


,dtype
Bank ID,string[python]
Account Number,string[python]
Entity ID,string[python]


## 18. 계좌 복합키 중복 확인

In [57]:
for ratio in RATIOS:
    accounts = processed_samples[ratio]["accounts"]

    composite_duplicate_mask = accounts.duplicated(
        subset=["Bank ID", "Account Number"],
        keep=False,
    )

    composite_duplicates = (
        accounts.loc[composite_duplicate_mask]
        .sort_values(["Bank ID", "Account Number"])
        .reset_index(drop=True)
    )

    print(f"\n===== {ratio}-{SIZE} 계좌 복합키 검사 =====")
    print(
        "복합키 중복 관련 전체 행 수:",
        len(composite_duplicates),
    )
    print(
        "첫 행을 제외한 중복 행 수:",
        accounts.duplicated(
            subset=["Bank ID", "Account Number"]
        ).sum(),
    )

    display(composite_duplicates)


===== HI-Small 계좌 복합키 검사 =====
복합키 중복 관련 전체 행 수: 0
첫 행을 제외한 중복 행 수: 0


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name



===== LI-Small 계좌 복합키 검사 =====
복합키 중복 관련 전체 행 수: 0
첫 행을 제외한 중복 행 수: 0


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name


## 19. 거래 계좌가 accounts에 존재하는지 검사

In [96]:
account_match_results = {}

def normalize_bank_id(series):
    normalized = (
        series
        .astype("string")
        .str.lstrip("0")
    )

    return normalized.mask(
        normalized.eq(""),
        "0",
    )


account_match_results = {}

for ratio in RATIOS:
    transactions = (
        processed_samples[ratio]["transactions"]
    )
    accounts = (
        processed_samples[ratio]["accounts"]
    )

    # accounts의 복합 계좌키
    account_keys = pd.MultiIndex.from_frame(
        pd.DataFrame({
            "bank": normalize_bank_id(
                accounts["Bank ID"]
            ),
            "account": (
                accounts["Account Number"]
                .astype("string")
            ),
        })
        .drop_duplicates()
    )

    # 송금 계좌 복합키
    sender_keys = pd.MultiIndex.from_arrays([
        normalize_bank_id(
            transactions["From Bank"]
        ),
        transactions["Account"].astype("string"),
    ])

    # 수취 계좌 복합키
    receiver_keys = pd.MultiIndex.from_arrays([
        normalize_bank_id(
            transactions["To Bank"]
        ),
        transactions["Account.1"].astype("string"),
    ])

    sender_missing = int(
        (~sender_keys.isin(account_keys)).sum()
    )

    receiver_missing = int(
        (~receiver_keys.isin(account_keys)).sum()
    )

    account_match_results[ratio] = {
        "sender_missing": sender_missing,
        "receiver_missing": receiver_missing,
    }

    print(
        f"\n===== {ratio}-{SIZE} "
        "계좌 연결 검사 ====="
    )
    print(
        "송금 계좌 미등록 거래:",
        f"{sender_missing:,}",
    )
    print(
        "수취 계좌 미등록 거래:",
        f"{receiver_missing:,}",
    )
    print(
        "송금 계좌 미등록 비율:",
        f"{sender_missing / len(transactions) * 100:.6f}%",
    )
    print(
        "수취 계좌 미등록 비율:",
        f"{receiver_missing / len(transactions) * 100:.6f}%",
    )


===== HI-Small 계좌 연결 검사 =====
송금 계좌 미등록 거래: 0
수취 계좌 미등록 거래: 0
송금 계좌 미등록 비율: 0.000000%
수취 계좌 미등록 비율: 0.000000%

===== LI-Small 계좌 연결 검사 =====
송금 계좌 미등록 거래: 0
수취 계좌 미등록 거래: 0
송금 계좌 미등록 비율: 0.000000%
수취 계좌 미등록 비율: 0.000000%


##

# 2. 데이터 분석

## 1. 패턴 파일 파싱 함수

In [59]:
import csv
from io import StringIO

import pandas as pd


PATTERN_TRANSACTION_COLUMNS = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]


def parse_pattern_file(path, ratio):
    """
    IBM AML Patterns.txt 파일을 읽어
    거래 한 행당 하나의 레코드로 변환한다.

    반환 컬럼
    ---------
    Dataset
    Attempt ID
    Attempt Number
    Pattern Type
    Pattern Meta
    Transaction Order
    기존 거래 컬럼 11개
    """

    records = []

    current_attempt_id = None
    current_attempt_number = None
    current_pattern_type = None
    current_pattern_meta = None
    transaction_order = 0

    attempt_number = 0

    with open(
        path,
        mode="r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line_number, raw_line in enumerate(
            file,
            start=1,
        ):
            line = raw_line.strip()

            # 빈 줄은 건너뜀
            if not line:
                continue

            begin_prefix = (
                "BEGIN LAUNDERING ATTEMPT - "
            )

            end_prefix = (
                "END LAUNDERING ATTEMPT - "
            )

            # 새로운 attempt 시작
            if line.startswith(begin_prefix):
                if current_attempt_id is not None:
                    raise ValueError(
                        "이전 attempt가 종료되기 전에 "
                        f"새 attempt가 시작됨: {line_number}행"
                    )

                attempt_number += 1

                pattern_text = line[
                    len(begin_prefix):
                ].strip()

                # 예:
                # FAN-OUT: Max 16-degree Fan-Out
                pattern_parts = pattern_text.split(
                    ":",
                    maxsplit=1,
                )

                current_pattern_type = (
                    pattern_parts[0]
                    .strip()
                    .upper()
                )

                if len(pattern_parts) == 2:
                    current_pattern_meta = (
                        pattern_parts[1].strip()
                    )
                else:
                    current_pattern_meta = None

                current_attempt_number = (
                    attempt_number
                )

                current_attempt_id = (
                    f"{ratio}_{attempt_number:04d}"
                )

                transaction_order = 0

                continue

            # 현재 attempt 종료
            if line.startswith(end_prefix):
                if current_attempt_id is None:
                    raise ValueError(
                        "시작되지 않은 attempt가 종료됨: "
                        f"{line_number}행"
                    )

                end_pattern_type = (
                    line[len(end_prefix):]
                    .strip()
                    .upper()
                )

                if (
                    end_pattern_type
                    != current_pattern_type
                ):
                    raise ValueError(
                        "BEGIN과 END의 패턴명이 다름: "
                        f"{line_number}행, "
                        f"{current_pattern_type} != "
                        f"{end_pattern_type}"
                    )

                current_attempt_id = None
                current_attempt_number = None
                current_pattern_type = None
                current_pattern_meta = None
                transaction_order = 0

                continue

            # BEGIN과 END 밖에 거래 행이 있으면 오류
            if current_attempt_id is None:
                raise ValueError(
                    "attempt 밖에서 거래 행 발견: "
                    f"{line_number}행"
                )

            # CSV 형식의 거래 한 줄 파싱
            parsed_row = next(
                csv.reader(
                    StringIO(line)
                )
            )

            if (
                len(parsed_row)
                != len(PATTERN_TRANSACTION_COLUMNS)
            ):
                raise ValueError(
                    f"{line_number}행의 컬럼 수가 "
                    f"{len(parsed_row)}개임. "
                    f"예상값은 "
                    f"{len(PATTERN_TRANSACTION_COLUMNS)}개"
                )

            transaction_order += 1

            transaction_record = dict(
                zip(
                    PATTERN_TRANSACTION_COLUMNS,
                    parsed_row,
                )
            )

            records.append({
                "Dataset": ratio,
                "Attempt ID": current_attempt_id,
                "Attempt Number": (
                    current_attempt_number
                ),
                "Pattern Type": (
                    current_pattern_type
                ),
                "Pattern Meta": (
                    current_pattern_meta
                ),
                "Transaction Order": (
                    transaction_order
                ),
                **transaction_record,
            })

    # 마지막 attempt가 종료되지 않은 경우
    if current_attempt_id is not None:
        raise ValueError(
            "파일 마지막 attempt에 END가 없습니다: "
            f"{current_attempt_id}"
        )

    pattern_df = pd.DataFrame(records)

    return pattern_df

## 2. HI와 LI 패턴 파일 파싱

In [60]:
pattern_transactions = {}

for ratio in RATIOS:
    pattern_df = parse_pattern_file(
        path=dataset_paths[ratio]["patterns"],
        ratio=ratio,
    )

    # Timestamp만 datetime으로 변환
    pattern_df["Timestamp"] = pd.to_datetime(
        pattern_df["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="coerce",
    )

    # 금액 컬럼 변환
    amount_columns = [
        "Amount Received",
        "Amount Paid",
    ]

    pattern_df[amount_columns] = (
        pattern_df[amount_columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    # 라벨 변환
    pattern_df["Is Laundering"] = (
        pd.to_numeric(
            pattern_df["Is Laundering"],
            errors="coerce",
        )
        .astype("Int8")
    )

    # ID 컬럼은 문자열로 명시
    id_columns = [
        "From Bank",
        "Account",
        "To Bank",
        "Account.1",
    ]

    pattern_df[id_columns] = (
        pattern_df[id_columns]
        .astype("string")
    )

    pattern_transactions[ratio] = pattern_df

    print(f"\n===== {ratio}-{SIZE} 패턴 파싱 결과 =====")
    print("거래 행 수:", len(pattern_df))
    print(
        "Attempt 수:",
        pattern_df["Attempt ID"].nunique(),
    )
    print(
        "패턴 종류 수:",
        pattern_df["Pattern Type"].nunique(),
    )

    display(pattern_df.head())


===== HI-Small 패턴 파싱 결과 =====
거래 행 수: 3209
Attempt 수: 370
패턴 종류 수: 8


,Dataset,Attempt ID,Attempt Number,Pattern Type,Pattern Meta,Transaction Order,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,HI,HI_0001,1,FAN-OUT,Max 16-degree Fan-Out,1,2022-09-01 00:06:00,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
1,HI,HI_0001,1,FAN-OUT,Max 16-degree Fan-Out,2,2022-09-01 04:33:00,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2,HI,HI_0001,1,FAN-OUT,Max 16-degree Fan-Out,3,2022-09-01 09:14:00,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
3,HI,HI_0001,1,FAN-OUT,Max 16-degree Fan-Out,4,2022-09-01 09:56:00,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
4,HI,HI_0001,1,FAN-OUT,Max 16-degree Fan-Out,5,2022-09-01 11:28:00,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1



===== LI-Small 패턴 파싱 결과 =====
거래 행 수: 1023
Attempt 수: 117
패턴 종류 수: 8


,Dataset,Attempt ID,Attempt Number,Pattern Type,Pattern Meta,Transaction Order,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,LI,LI_0001,1,FAN-IN,Max 3-degree Fan-In,1,2022-09-01 02:38:00,001812,80279F810,0110,8000A94C0,10154.74,Australian Dollar,10154.74,Australian Dollar,ACH,1
1,LI,LI_0001,1,FAN-IN,Max 3-degree Fan-In,2,2022-09-02 14:36:00,022595,80279F8B0,0110,8000A94C0,5326.79,Australian Dollar,5326.79,Australian Dollar,ACH,1
2,LI,LI_0001,1,FAN-IN,Max 3-degree Fan-In,3,2022-09-03 14:09:00,001120,800E36A50,0110,8000A94C0,4634.81,Australian Dollar,4634.81,Australian Dollar,ACH,1
3,LI,LI_0002,2,FAN-IN,Max 8-degree Fan-In,1,2022-09-01 03:17:00,003671,801BF8E70,002557,8016B3750,8099.96,Euro,8099.96,Euro,ACH,1
4,LI,LI_0002,2,FAN-IN,Max 8-degree Fan-In,2,2022-09-01 06:27:00,015,80074C7E0,002557,8016B3750,10468.56,Euro,10468.56,Euro,ACH,1


## 3. 파싱 결과 검증

In [61]:
for ratio in RATIOS:
    pattern_df = pattern_transactions[ratio]

    print(f"\n===== {ratio}-{SIZE} 파싱 검증 =====")

    validation = pd.Series({
        "전체 거래 행 수": len(pattern_df),

        "Attempt 수": (
            pattern_df["Attempt ID"].nunique()
        ),

        "패턴 종류 수": (
            pattern_df["Pattern Type"].nunique()
        ),

        "Timestamp 변환 실패": (
            pattern_df["Timestamp"].isna().sum()
        ),

        "Amount Received 변환 실패": (
            pattern_df[
                "Amount Received"
            ].isna().sum()
        ),

        "Amount Paid 변환 실패": (
            pattern_df[
                "Amount Paid"
            ].isna().sum()
        ),

        "Is Laundering 변환 실패": (
            pattern_df[
                "Is Laundering"
            ].isna().sum()
        ),

        "라벨이 1이 아닌 거래 수": (
            pattern_df[
                "Is Laundering"
            ].ne(1).sum()
        ),
    })

    display(validation.to_frame("value"))


===== HI-Small 파싱 검증 =====


,value
전체 거래 행 수,3209
Attempt 수,370
패턴 종류 수,8
Timestamp 변환 실패,0
Amount Received 변환 실패,0
Amount Paid 변환 실패,0
Is Laundering 변환 실패,0
라벨이 1이 아닌 거래 수,0



===== LI-Small 파싱 검증 =====


,value
전체 거래 행 수,1023
Attempt 수,117
패턴 종류 수,8
Timestamp 변환 실패,0
Amount Received 변환 실패,0
Amount Paid 변환 실패,0
Is Laundering 변환 실패,0
라벨이 1이 아닌 거래 수,0


## 4. 패턴별 attempt와 거래 수 확인

In [62]:
pattern_summary_rows = []

for ratio in RATIOS:
    pattern_df = pattern_transactions[ratio]

    summary = (
        pattern_df
        .groupby("Pattern Type")
        .agg(
            attempt_count=(
                "Attempt ID",
                "nunique",
            ),
            transaction_count=(
                "Attempt ID",
                "size",
            ),
            min_transactions_per_attempt=(
                "Attempt ID",
                lambda x: (
                    x.value_counts().min()
                ),
            ),
            median_transactions_per_attempt=(
                "Attempt ID",
                lambda x: (
                    x.value_counts().median()
                ),
            ),
            max_transactions_per_attempt=(
                "Attempt ID",
                lambda x: (
                    x.value_counts().max()
                ),
            ),
        )
        .reset_index()
    )

    summary.insert(0, "Dataset", ratio)
    pattern_summary_rows.append(summary)

pattern_summary = pd.concat(
    pattern_summary_rows,
    ignore_index=True,
)

display(pattern_summary)

,Dataset,Pattern Type,attempt_count,transaction_count,min_transactions_per_attempt,median_transactions_per_attempt,max_transactions_per_attempt
0,HI,BIPARTITE,49,263,1,4.0,15
1,HI,CYCLE,54,287,2,4.0,12
2,HI,FAN-IN,40,318,1,8.0,16
3,HI,FAN-OUT,48,342,1,7.0,16
4,HI,GATHER-SCATTER,51,716,2,14.0,28
5,HI,RANDOM,41,191,1,3.0,11
6,HI,SCATTER-GATHER,44,626,2,14.0,32
7,HI,STACK,43,466,2,10.0,30
8,LI,BIPARTITE,16,129,1,8.0,14
9,LI,CYCLE,12,83,2,5.5,13


In [63]:
attempt_summary_rows = []

for ratio in RATIOS:
    pattern_df = pattern_transactions[ratio]

    attempt_summary = (
        pattern_df
        .groupby(
            [
                "Dataset",
                "Attempt ID",
                "Attempt Number",
                "Pattern Type",
                "Pattern Meta",
            ],
            dropna=False,
        )
        .agg(
            transaction_count=(
                "Transaction Order",
                "size",
            ),
            node_count=(
                "Account",
                lambda x: 0,
            ),
            start_time=(
                "Timestamp",
                "min",
            ),
            end_time=(
                "Timestamp",
                "max",
            ),
            receiving_currency_count=(
                "Receiving Currency",
                "nunique",
            ),
            payment_currency_count=(
                "Payment Currency",
                "nunique",
            ),
        )
        .reset_index()
    )

    attempt_summary["duration"] = (
        attempt_summary["end_time"]
        - attempt_summary["start_time"]
    )

    attempt_summary_rows.append(
        attempt_summary
    )

attempt_summary = pd.concat(
    attempt_summary_rows,
    ignore_index=True,
)

display(attempt_summary.head())

,Dataset,Attempt ID,Attempt Number,Pattern Type,Pattern Meta,transaction_count,node_count,start_time,end_time,receiving_currency_count,payment_currency_count,duration
0,HI,HI_0001,1,FAN-OUT,Max 16-degree Fan-Out,16,0,2022-09-01 00:06:00,2022-09-04 15:48:00,3,3,3 days 15:42:00
1,HI,HI_0002,2,CYCLE,Max 10 hops,10,0,2022-09-01 00:03:00,2022-09-04 15:51:00,9,9,3 days 15:48:00
2,HI,HI_0003,3,GATHER-SCATTER,Max 3-degree Fan-In,5,0,2022-09-01 00:04:00,2022-09-04 14:59:00,1,1,3 days 14:55:00
3,HI,HI_0004,4,STACK,NaN,22,0,2022-09-01 08:40:00,2022-09-04 19:11:00,7,7,3 days 10:31:00
4,HI,HI_0005,5,GATHER-SCATTER,Max 13-degree Fan-In,17,0,2022-09-01 00:43:00,2022-09-06 19:16:00,3,3,5 days 18:33:00


## 5. 우선 STACK 하나 열어보기

In [64]:
for ratio in RATIOS:
    pattern_df = pattern_transactions[ratio]

    stack_attempt_ids = (
        pattern_df.loc[
            pattern_df["Pattern Type"].eq("STACK"),
            "Attempt ID",
        ]
        .drop_duplicates()
        .tolist()
    )

    first_stack_id = stack_attempt_ids[0]

    first_stack = (
        pattern_df.loc[
            pattern_df["Attempt ID"].eq(
                first_stack_id
            )
        ]
        .sort_values(
            [
                "Timestamp",
                "Transaction Order",
            ]
        )
        .reset_index(drop=True)
    )

    print(
        f"\n===== {ratio}-{SIZE}: "
        f"{first_stack_id} ====="
    )

    display(
        first_stack[
            [
                "Transaction Order",
                "Timestamp",
                "From Bank",
                "Account",
                "To Bank",
                "Account.1",
                "Amount Paid",
                "Payment Currency",
                "Amount Received",
                "Receiving Currency",
                "Payment Format",
            ]
        ]
    )


===== HI-Small: HI_0004 =====


,Transaction Order,Timestamp,From Bank,Account,To Bank,Account.1,Amount Paid,Payment Currency,Amount Received,Receiving Currency,Payment Format
0,19,2022-09-01 08:40:00,0214,80B7347A0,00410,802D878C0,6702.62,Euro,6702.62,Euro,ACH
1,5,2022-09-01 09:22:00,0040836,80F6B88B0,0016606,8064545E0,11800.69,US Dollar,11800.69,US Dollar,ACH
2,3,2022-09-01 10:19:00,018617,8038D3520,024482,801C0F2B0,13712.96,Euro,13712.96,Euro,ACH
3,11,2022-09-01 14:28:00,010,8009CF210,011318,8011AB110,11964.97,US Dollar,11964.97,US Dollar,ACH
4,9,2022-09-01 15:05:00,023842,80195DE20,002845,801E2BC20,277.59,US Dollar,277.59,US Dollar,ACH
5,21,2022-09-01 16:51:00,02591,80A1C7710,024850,80A09CF50,3724.19,Euro,3724.19,Euro,ACH
6,17,2022-09-01 18:17:00,022806,807A61380,006179,802D53D80,16345.56,US Dollar,16345.56,US Dollar,ACH
7,7,2022-09-02 07:21:00,021749,8014BDC90,0214050,80537F610,317448.19,Rupee,317448.19,Rupee,ACH
8,15,2022-09-02 07:37:00,0115700,80B3E96E0,014099,801F8CCB0,17684.19,US Dollar,17684.19,US Dollar,ACH
9,1,2022-09-02 12:36:00,0223,8000DD890,0040312,80F52D550,14153.46,Swiss Franc,14153.46,Swiss Franc,ACH



===== LI-Small: LI_0010 =====


,Transaction Order,Timestamp,From Bank,Account,To Bank,Account.1,Amount Paid,Payment Currency,Amount Received,Receiving Currency,Payment Format
0,5,2022-09-03 08:45:00,000,80023E6D0,020,80025F000,774.62,Euro,774.62,Euro,ACH
1,1,2022-09-03 09:30:00,003,8001B12A0,001439,800A35E30,7757.88,US Dollar,7757.88,US Dollar,ACH
2,3,2022-09-03 13:05:00,02860,800E66D90,001110,8008622A0,16196.46,US Dollar,16196.46,US Dollar,ACH
3,6,2022-09-04 07:47:00,020,80025F000,018,800076370,94401.67,Yen,94401.67,Yen,ACH
4,4,2022-09-05 04:18:00,001110,8008622A0,01768,800484FC0,13533.23,Euro,13533.23,Euro,ACH
5,2,2022-09-05 09:43:00,001439,800A35E30,00423,800340AE0,6254.47,Euro,6254.47,Euro,ACH


## 6. 패턴외 자금세탁 알아보기

In [65]:
transaction_key_columns = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

In [66]:
pattern_match_results = {}

for ratio in RATIOS:
    transactions = (
        processed_samples[ratio]["transactions"]
    )

    patterns = pattern_transactions[ratio]

    # Trans.csv에 있는 자금세탁 거래
    laundering_transactions = (
        transactions.loc[
            transactions["Is Laundering"].eq(1),
            transaction_key_columns,
        ]
        .copy()
    )

    # Patterns.txt에 있는 거래
    pattern_rows = (
        patterns[
            transaction_key_columns
        ]
        .copy()
    )

    # 같은 거래가 여러 번 있더라도
    # 우선 거래 존재 여부만 비교
    laundering_unique = (
        laundering_transactions
        .drop_duplicates()
    )

    pattern_unique = (
        pattern_rows
        .drop_duplicates()
    )

    # Trans의 자금세탁 거래가 Patterns에 존재하는지
    trans_check = laundering_unique.merge(
        pattern_unique,
        how="left",
        on=transaction_key_columns,
        indicator=True,
    )

    trans_only = (
        trans_check.loc[
            trans_check["_merge"].eq("left_only")
        ]
        .drop(columns="_merge")
        .reset_index(drop=True)
    )

    # Patterns 거래가 Trans에 존재하는지
    pattern_check = pattern_unique.merge(
        laundering_unique,
        how="left",
        on=transaction_key_columns,
        indicator=True,
    )

    pattern_only = (
        pattern_check.loc[
            pattern_check["_merge"].eq("left_only")
        ]
        .drop(columns="_merge")
        .reset_index(drop=True)
    )

    matched_count = (
        trans_check["_merge"]
        .eq("both")
        .sum()
    )

    pattern_match_results[ratio] = {
        "trans_only": trans_only,
        "pattern_only": pattern_only,
    }

    print(f"\n===== {ratio}-{SIZE} 패턴 연결 검사 =====")
    print(
        "Trans의 자금세탁 거래 수:",
        len(laundering_transactions),
    )
    print(
        "Trans의 고유 자금세탁 거래 수:",
        len(laundering_unique),
    )
    print(
        "Patterns의 거래 수:",
        len(pattern_rows),
    )
    print(
        "Patterns의 고유 거래 수:",
        len(pattern_unique),
    )
    print(
        "양쪽에서 일치한 거래 수:",
        matched_count,
    )
    print(
        "Trans에만 있는 자금세탁 거래 수:",
        len(trans_only),
    )
    print(
        "Patterns에만 있는 거래 수:",
        len(pattern_only),
    )


===== HI-Small 패턴 연결 검사 =====
Trans의 자금세탁 거래 수: 5177
Trans의 고유 자금세탁 거래 수: 5177
Patterns의 거래 수: 3209
Patterns의 고유 거래 수: 3209
양쪽에서 일치한 거래 수: 3209
Trans에만 있는 자금세탁 거래 수: 1968
Patterns에만 있는 거래 수: 0

===== LI-Small 패턴 연결 검사 =====
Trans의 자금세탁 거래 수: 3565
Trans의 고유 자금세탁 거래 수: 3565
Patterns의 거래 수: 1023
Patterns의 고유 거래 수: 1023
양쪽에서 일치한 거래 수: 1023
Trans에만 있는 자금세탁 거래 수: 2542
Patterns에만 있는 거래 수: 0


In [67]:
for ratio in RATIOS:
    trans_only = (
        pattern_match_results[ratio]["trans_only"]
    )

    print(
        f"\n===== {ratio}-{SIZE} "
        "패턴에 없는 자금세탁 거래 ====="
    )

    display(trans_only.head(100))


===== HI-Small 패턴에 없는 자금세탁 거래 =====


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:21:00,070,100428660,001124,800825340,3.897694e+05,US Dollar,3.897694e+05,US Dollar,Cheque,1
1,2022-09-01 00:03:00,070,100428660,011474,805B716C0,2.902433e+04,US Dollar,2.902433e+04,US Dollar,Credit Card,1
2,2022-09-01 00:01:00,070,100428660,015980,80B39E7B0,7.929200e+02,US Dollar,7.929200e+02,US Dollar,Credit Card,1
3,2022-09-01 00:03:00,070,100428660,0113798,80DC756E0,1.317143e+07,US Dollar,1.317143e+07,US Dollar,Cheque,1
4,2022-09-01 00:23:00,070,100428660,0032375,80E480620,1.428883e+04,US Dollar,1.428883e+04,US Dollar,Cash,1
...,...,...,...,...,...,...,...,...,...,...,...
95,2022-09-01 09:05:00,070,100428A51,0254242,813CC3021,1.729459e+00,Bitcoin,1.729459e+00,Bitcoin,Bitcoin,1
96,2022-09-01 09:39:00,070,1004286F0,009417,814422CF0,1.150800e+03,Yuan,1.150800e+03,Yuan,Cash,1
97,2022-09-01 09:46:00,070,100428738,019925,803FA2490,2.460108e+04,Yen,2.460108e+04,Yen,Credit Card,1
98,2022-09-01 09:42:00,0024779,8090EE1D0,008,8090EF0B0,5.780560e+03,Canadian Dollar,5.780560e+03,Canadian Dollar,ACH,1



===== LI-Small 패턴에 없는 자금세탁 거래 =====


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:02:00,070,10042B660,022661,805F7F2B0,7.083164e+04,US Dollar,7.083164e+04,US Dollar,Cash,1
1,2022-09-01 00:19:00,070,10042B660,0212996,80609B4C0,3.370547e+04,US Dollar,3.370547e+04,US Dollar,Cash,1
2,2022-09-01 00:01:00,070,10042B660,0011305,807861770,1.097976e+06,US Dollar,1.097976e+06,US Dollar,Cash,1
3,2022-09-01 00:00:00,011968,815630C40,0249349,815635220,8.923300e+02,US Dollar,8.923300e+02,US Dollar,ACH,1
4,2022-09-01 00:25:00,070,10042B660,011968,816F93AF0,1.808140e+03,US Dollar,1.808140e+03,US Dollar,Cheque,1
...,...,...,...,...,...,...,...,...,...,...,...
95,2022-09-01 07:03:00,0024816,80A769B80,0024816,80A769FE0,2.960437e+04,Rupee,2.960437e+04,Rupee,ACH,1
96,2022-09-01 07:23:00,070,10042B780,0025520,814F477D0,4.286015e+06,Rupee,4.286015e+06,Rupee,Cheque,1
97,2022-09-01 07:26:00,070,10042B930,0010,811EDCC10,1.044519e+05,Brazil Real,1.044519e+05,Brazil Real,Credit Card,1
98,2022-09-01 07:03:00,0063779,818C3F850,0264772,818C420A0,1.598899e+04,Saudi Riyal,1.598899e+04,Saudi Riyal,ACH,1


In [68]:
for ratio in RATIOS:
    pattern_only = (
        pattern_match_results[ratio]["pattern_only"]
    )

    print(
        f"\n===== {ratio}-{SIZE} "
        "Trans에 연결되지 않은 패턴 거래 ====="
    )

    display(pattern_only.head(100))


===== HI-Small Trans에 연결되지 않은 패턴 거래 =====


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering



===== LI-Small Trans에 연결되지 않은 패턴 거래 =====


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering


## 7. 단건 거래 분석

In [69]:
import gc

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.csgraph import connected_components

In [70]:
TRANSACTION_KEY_COLUMNS = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]


def build_transaction_graph_index(transactions):
    """
    계좌를 노드, 거래를 방향성 엣지로 변환한다.

    연결요소 계산에서는 거래 방향을 무시한
    weakly connected component를 사용한다.
    """

    sender_nodes = (
        transactions["From Bank"]
        .astype("string")
        .str.cat(
            transactions["Account"].astype("string"),
            sep="::",
        )
    )

    receiver_nodes = (
        transactions["To Bank"]
        .astype("string")
        .str.cat(
            transactions["Account.1"].astype("string"),
            sep="::",
        )
    )

    transaction_count = len(transactions)

    all_nodes = pd.concat(
        [
            sender_nodes,
            receiver_nodes,
        ],
        ignore_index=True,
    )

    node_codes, node_values = pd.factorize(
        all_nodes,
        sort=False,
    )

    # Small 데이터셋에서는 int32 범위로 충분
    node_codes = node_codes.astype(
        np.int32,
        copy=False,
    )

    source_codes = node_codes[
        :transaction_count
    ]

    target_codes = node_codes[
        transaction_count:
    ]

    del all_nodes
    del node_codes
    del sender_nodes
    del receiver_nodes

    node_count = len(node_values)

    # 동일 계좌 쌍의 반복 거래는 희소행렬에서 합쳐짐
    graph = sparse.coo_matrix(
        (
            np.ones(
                transaction_count,
                dtype=np.uint8,
            ),
            (
                source_codes,
                target_codes,
            ),
        ),
        shape=(node_count, node_count),
    ).tocsr()

    graph.sum_duplicates()
    graph.data[:] = 1

    component_count, node_components = (
        connected_components(
            graph,
            directed=True,
            connection="weak",
        )
    )

    del graph
    gc.collect()

    laundering_labels = (
        transactions["Is Laundering"]
        .to_numpy(dtype=np.int8)
    )

    # 같은 거래의 송·수취 노드는 동일한 weak component
    edge_components = node_components[
        source_codes
    ]

    component_transaction_count = np.bincount(
        edge_components,
        minlength=component_count,
    )

    component_laundering_count = np.bincount(
        edge_components,
        weights=laundering_labels,
        minlength=component_count,
    ).astype(np.int64)

    component_node_count = np.bincount(
        node_components,
        minlength=component_count,
    )

    normal_only_components = (
        component_laundering_count == 0
    )

    return {
        "source_codes": source_codes,
        "target_codes": target_codes,
        "node_values": node_values,
        "node_components": node_components,
        "edge_components": edge_components,
        "labels": laundering_labels,
        "component_count": component_count,
        "component_transaction_count": (
            component_transaction_count
        ),
        "component_laundering_count": (
            component_laundering_count
        ),
        "component_node_count": (
            component_node_count
        ),
        "normal_only_components": (
            normal_only_components
        ),
    }

In [71]:
def summarize_graph_components(graph_index):
    component_count = (
        graph_index["component_count"]
    )

    component_transaction_count = (
        graph_index[
            "component_transaction_count"
        ]
    )

    component_laundering_count = (
        graph_index[
            "component_laundering_count"
        ]
    )

    component_node_count = (
        graph_index[
            "component_node_count"
        ]
    )

    normal_only = (
        graph_index["normal_only_components"]
    )

    total_transactions = (
        component_transaction_count.sum()
    )

    total_nodes = component_node_count.sum()

    summary = pd.DataFrame([
        {
            "component_type": "전체",
            "component_count": component_count,
            "account_count": total_nodes,
            "transaction_count": total_transactions,
            "laundering_count": (
                component_laundering_count.sum()
            ),
        },
        {
            "component_type": "정상 거래만 존재",
            "component_count": normal_only.sum(),
            "account_count": (
                component_node_count[
                    normal_only
                ].sum()
            ),
            "transaction_count": (
                component_transaction_count[
                    normal_only
                ].sum()
            ),
            "laundering_count": 0,
        },
        {
            "component_type": "자금세탁 거래 포함",
            "component_count": (
                (~normal_only).sum()
            ),
            "account_count": (
                component_node_count[
                    ~normal_only
                ].sum()
            ),
            "transaction_count": (
                component_transaction_count[
                    ~normal_only
                ].sum()
            ),
            "laundering_count": (
                component_laundering_count[
                    ~normal_only
                ].sum()
            ),
        },
    ])

    summary[
        "component_ratio_percent"
    ] = (
        summary["component_count"]
        / component_count
        * 100
    )

    summary[
        "account_ratio_percent"
    ] = (
        summary["account_count"]
        / total_nodes
        * 100
    )

    summary[
        "transaction_ratio_percent"
    ] = (
        summary["transaction_count"]
        / total_transactions
        * 100
    )

    component_detail = pd.DataFrame({
        "component_id": np.arange(
            component_count
        ),
        "account_count": (
            component_node_count
        ),
        "transaction_count": (
            component_transaction_count
        ),
        "laundering_count": (
            component_laundering_count
        ),
        "normal_only": normal_only,
    })

    largest_normal_components = (
        component_detail.loc[
            component_detail["normal_only"]
        ]
        .nlargest(
            20,
            [
                "transaction_count",
                "account_count",
            ],
        )
        .reset_index(drop=True)
    )

    return summary, largest_normal_components

In [72]:
def normalized_transaction_hash(frame):
    """
    Trans와 Patterns의 dtype 차이를 제거한 뒤
    동일 거래를 비교하기 위한 해시를 생성한다.
    """

    normalized = pd.DataFrame({
        "Timestamp": (
            pd.to_datetime(
                frame["Timestamp"]
            ).astype("int64")
        ),
        "From Bank": (
            frame["From Bank"]
            .astype("string")
        ),
        "Account": (
            frame["Account"]
            .astype("string")
        ),
        "To Bank": (
            frame["To Bank"]
            .astype("string")
        ),
        "Account.1": (
            frame["Account.1"]
            .astype("string")
        ),
        "Amount Received": (
            pd.to_numeric(
                frame["Amount Received"]
            ).astype("float64")
        ),
        "Receiving Currency": (
            frame["Receiving Currency"]
            .astype("string")
        ),
        "Amount Paid": (
            pd.to_numeric(
                frame["Amount Paid"]
            ).astype("float64")
        ),
        "Payment Currency": (
            frame["Payment Currency"]
            .astype("string")
        ),
        "Payment Format": (
            frame["Payment Format"]
            .astype("string")
        ),
        "Is Laundering": (
            pd.to_numeric(
                frame["Is Laundering"]
            ).astype("int8")
        ),
    })

    return (
        pd.util.hash_pandas_object(
            normalized,
            index=False,
        )
        .to_numpy(dtype=np.uint64)
    )

In [73]:
def find_single_laundering_rows(
    transactions,
    patterns,
    graph_index,
):
    """
    1. 거래 1건짜리 Patterns attempt
    2. 패턴 외 자금세탁 연결요소 중 거래 1건짜리 블록

    에 해당하는 Trans 행 인덱스를 반환한다.
    """

    labels = graph_index["labels"]

    source_codes = (
        graph_index["source_codes"]
    )

    target_codes = (
        graph_index["target_codes"]
    )

    laundering_rows = np.flatnonzero(
        labels == 1
    )

    laundering_frame = transactions.iloc[
        laundering_rows
    ]

    laundering_hashes = (
        normalized_transaction_hash(
            laundering_frame
        )
    )

    pattern_hashes = (
        normalized_transaction_hash(
            patterns
        )
    )

    # 거래가 1건인 generator attempt
    attempt_sizes = (
        patterns
        .groupby("Attempt ID")[
            "Attempt ID"
        ]
        .transform("size")
    )

    single_pattern_frame = patterns.loc[
        attempt_sizes.eq(1)
    ]

    single_pattern_hashes = set(
        normalized_transaction_hash(
            single_pattern_frame
        ).tolist()
    )

    all_pattern_hashes = set(
        pattern_hashes.tolist()
    )

    # Trans에서 단건 pattern attempt에 해당하는 행
    is_single_pattern = np.isin(
        laundering_hashes,
        list(single_pattern_hashes),
    )

    single_pattern_rows = laundering_rows[
        is_single_pattern
    ]

    # Patterns.txt에 포함되지 않은 자금세탁 거래
    is_outside = ~np.isin(
        laundering_hashes,
        list(all_pattern_hashes),
    )

    outside_rows = laundering_rows[
        is_outside
    ]

    if len(outside_rows) == 0:
        single_outside_rows = np.array(
            [],
            dtype=np.int64,
        )
    else:
        outside_sources = source_codes[
            outside_rows
        ]

        outside_targets = target_codes[
            outside_rows
        ]

        outside_node_values, inverse = np.unique(
            np.concatenate([
                outside_sources,
                outside_targets,
            ]),
            return_inverse=True,
        )

        outside_count = len(outside_rows)

        local_sources = inverse[
            :outside_count
        ]

        local_targets = inverse[
            outside_count:
        ]

        outside_graph = sparse.coo_matrix(
            (
                np.ones(
                    outside_count,
                    dtype=np.uint8,
                ),
                (
                    local_sources,
                    local_targets,
                ),
            ),
            shape=(
                len(outside_node_values),
                len(outside_node_values),
            ),
        ).tocsr()

        (
            outside_component_count,
            outside_node_components,
        ) = connected_components(
            outside_graph,
            directed=True,
            connection="weak",
        )

        outside_edge_components = (
            outside_node_components[
                local_sources
            ]
        )

        outside_component_edges = np.bincount(
            outside_edge_components,
            minlength=outside_component_count,
        )

        is_single_outside = (
            outside_component_edges[
                outside_edge_components
            ]
            == 1
        )

        single_outside_rows = outside_rows[
            is_single_outside
        ]

    single_rows = np.unique(
        np.concatenate([
            single_pattern_rows,
            single_outside_rows,
        ])
    )

    detail = pd.Series({
        "single_pattern_transactions": (
            len(single_pattern_rows)
        ),
        "single_outside_transactions": (
            len(single_outside_rows)
        ),
        "total_single_laundering": (
            len(single_rows)
        ),
    })

    return single_rows, detail

In [74]:
def scenario_statistics(
    name,
    remove_mask,
    graph_index,
):
    labels = graph_index["labels"]
    source_codes = graph_index["source_codes"]
    target_codes = graph_index["target_codes"]

    keep_mask = ~remove_mask

    kept_labels = labels[keep_mask]

    laundering_count = int(
        kept_labels.sum()
    )

    transaction_count = int(
        keep_mask.sum()
    )

    normal_count = (
        transaction_count
        - laundering_count
    )

    used_nodes = np.zeros(
        len(graph_index["node_values"]),
        dtype=bool,
    )

    used_nodes[
        source_codes[keep_mask]
    ] = True

    used_nodes[
        target_codes[keep_mask]
    ] = True

    return {
        "scenario": name,
        "account_count": int(
            used_nodes.sum()
        ),
        "transaction_count": (
            transaction_count
        ),
        "normal_count": normal_count,
        "laundering_count": (
            laundering_count
        ),
        "laundering_ratio_percent": (
            laundering_count
            / transaction_count
            * 100
            if transaction_count
            else np.nan
        ),
        "normal_per_laundering": (
            normal_count / laundering_count
            if laundering_count
            else np.inf
        ),
        "removed_transactions": int(
            remove_mask.sum()
        ),
        "removed_normal": int(
            labels[remove_mask].eq(0).sum()
        )
        if isinstance(labels, pd.Series)
        else int(
            (labels[remove_mask] == 0).sum()
        ),
        "removed_laundering": int(
            labels[remove_mask].sum()
        ),
    }

In [75]:
def compare_removal_scenarios(
    transactions,
    patterns,
    graph_index,
):
    labels = graph_index["labels"]

    source_codes = (
        graph_index["source_codes"]
    )

    target_codes = (
        graph_index["target_codes"]
    )

    transaction_count = len(labels)

    empty_remove_mask = np.zeros(
        transaction_count,
        dtype=bool,
    )

    # 정상 거래로만 이루어진 연결요소
    normal_only_edge_mask = (
        graph_index[
            "normal_only_components"
        ][
            graph_index["edge_components"]
        ]
    )

    single_rows, single_detail = (
        find_single_laundering_rows(
            transactions=transactions,
            patterns=patterns,
            graph_index=graph_index,
        )
    )

    # 단건 자금세탁 거래만 제거
    single_transaction_mask = np.zeros(
        transaction_count,
        dtype=bool,
    )

    single_transaction_mask[
        single_rows
    ] = True

    # 단건 거래의 양 끝 계좌
    seed_node_mask = np.zeros(
        len(graph_index["node_values"]),
        dtype=bool,
    )

    seed_node_mask[
        source_codes[single_rows]
    ] = True

    seed_node_mask[
        target_codes[single_rows]
    ] = True

    # 양 끝 계좌와 직접 연결된 모든 거래
    seed_incident_mask = (
        seed_node_mask[source_codes]
        | seed_node_mask[target_codes]
    )

    # 양 끝 계좌와 거래한 모든 1홉 계좌
    one_hop_node_mask = (
        seed_node_mask.copy()
    )

    one_hop_node_mask[
        source_codes[seed_incident_mask]
    ] = True

    one_hop_node_mask[
        target_codes[seed_incident_mask]
    ] = True

    # 1홉 정상 상대 계좌의 다른 거래까지 제거
    one_hop_incident_mask = (
        one_hop_node_mask[source_codes]
        | one_hop_node_mask[target_codes]
    )

    scenarios = [
        (
            "원본 전처리 데이터",
            empty_remove_mask,
        ),
        (
            "정상-only 고립 컴포넌트 제거",
            normal_only_edge_mask,
        ),
        (
            "단건 자금세탁 거래만 제거",
            single_transaction_mask,
        ),
        (
            "단건 자금세탁 계좌의 모든 거래 제거",
            seed_incident_mask,
        ),
        (
            "단건 거래의 1홉 계좌와 모든 거래 제거",
            one_hop_incident_mask,
        ),
        (
            "정상-only + 단건 계좌 거래 제거",
            (
                normal_only_edge_mask
                | seed_incident_mask
            ),
        ),
        (
            "정상-only + 1홉 계좌 전체 제거",
            (
                normal_only_edge_mask
                | one_hop_incident_mask
            ),
        ),
    ]

    scenario_table = pd.DataFrame([
        scenario_statistics(
            name=name,
            remove_mask=mask,
            graph_index=graph_index,
        )
        for name, mask in scenarios
    ])

    attached_summary = pd.Series({
        **single_detail.to_dict(),

        "single_seed_account_count": int(
            seed_node_mask.sum()
        ),

        "seed_incident_account_count": int(
            one_hop_node_mask.sum()
        ),

        "seed_incident_transaction_count": int(
            seed_incident_mask.sum()
        ),

        "seed_incident_normal_count": int(
            (
                seed_incident_mask
                & (labels == 0)
            ).sum()
        ),

        "seed_incident_laundering_count": int(
            labels[
                seed_incident_mask
            ].sum()
        ),

        "one_hop_incident_transaction_count": int(
            one_hop_incident_mask.sum()
        ),

        "one_hop_incident_normal_count": int(
            (
                one_hop_incident_mask
                & (labels == 0)
            ).sum()
        ),

        "one_hop_incident_laundering_count": int(
            labels[
                one_hop_incident_mask
            ].sum()
        ),
    })

    return scenario_table, attached_summary

In [77]:
imbalance_results = {}

for ratio in RATIOS:
    print(
        f"\n{'=' * 80}\n"
        f"{ratio}-{SIZE} 그래프 분석\n"
        f"{'=' * 80}"
    )

    transactions = (
        processed_samples[ratio][
            "transactions"
        ]
    )

    patterns = pattern_transactions[ratio]

    graph_index = (
        build_transaction_graph_index(
            transactions
        )
    )

    (
        component_summary,
        largest_normal_components,
    ) = summarize_graph_components(
        graph_index
    )

    (
        scenario_table,
        attached_summary,
    ) = compare_removal_scenarios(
        transactions=transactions,
        patterns=patterns,
        graph_index=graph_index,
    )

    print("\n[연결요소 구성]")
    display(component_summary)

    print("\n[가장 큰 정상-only 연결요소]")
    display(largest_normal_components)

    print("\n[단건 거래와 주변 계좌 규모]")
    display(
        attached_summary.to_frame("value")
    )

    print("\n[삭제 시나리오별 클래스 비율]")
    display(
        scenario_table.style.format({
            "laundering_ratio_percent": (
                "{:.6f}"
            ),
            "normal_per_laundering": (
                "{:.2f}"
            ),
        })
    )

    imbalance_results[ratio] = {
        "component_summary": (
            component_summary
        ),
        "largest_normal_components": (
            largest_normal_components
        ),
        "attached_summary": (
            attached_summary
        ),
        "scenario_table": scenario_table,
    }

    del graph_index
    gc.collect()


HI-Small 그래프 분석

[연결요소 구성]


,component_type,component_count,account_count,transaction_count,laundering_count,component_ratio_percent,account_ratio_percent,transaction_ratio_percent
0,전체,114139,515088,5078336,5177,100.000000,100.000000,100.000000
1,정상 거래만 존재,114131,142978,203737,0,99.992991,27.757975,4.011885
2,자금세탁 거래 포함,8,372110,4874599,5177,0.007009,72.242025,95.988115



[가장 큰 정상-only 연결요소]


,component_id,account_count,transaction_count,laundering_count,normal_only
0,8856,7,95,0,True
1,38724,23,87,0,True
2,44951,7,66,0,True
3,31142,5,62,0,True
4,13292,2,62,0,True
5,10112,4,60,0,True
6,609,5,58,0,True
7,1997,4,56,0,True
8,53861,3,56,0,True
9,53901,6,53,0,True



[단건 거래와 주변 계좌 규모]


...

,component_id,account_count,transaction_count,laundering_count,normal_only
0,51231,4,73,0,True
1,85097,3,67,0,True
2,17047,5,64,0,True
3,37096,8,58,0,True
4,3245,3,58,0,True
5,58065,4,56,0,True
6,77416,4,55,0,True
7,64664,8,54,0,True
8,61281,4,54,0,True
9,43244,3,54,0,True



[단건 거래와 주변 계좌 규모]


,value
single_pattern_transactions,8
single_outside_transactions,1586
total_single_laundering,1594
single_seed_account_count,3185
seed_incident_account_count,14576
seed_incident_transaction_count,60011
seed_incident_normal_count,58386
seed_incident_laundering_count,1625
one_hop_incident_transaction_count,864175
one_hop_incident_normal_count,861597



[삭제 시나리오별 클래스 비율]


,scenario,account_count,transaction_count,normal_count,laundering_count,laundering_ratio_percent,normal_per_laundering,removed_transactions,removed_normal,removed_laundering
0,원본 전처리 데이터,705907,6924041,6920476,3565,0.051487,1941.23,0,0,0
1,정상-only 고립 컴포넌트 제거,504418,6637998,6634433,3565,0.053706,1860.99,286043,286043,0
2,단건 자금세탁 거래만 제거,705867,6922447,6920476,1971,0.028473,3511.15,1594,0,1594
3,단건 자금세탁 계좌의 모든 거래 제거,701648,6864030,6862090,1940,0.028263,3537.16,60011,58386,1625
4,단건 거래의 1홉 계좌와 모든 거래 제거,686317,6059866,6058879,987,0.016287,6138.68,864175,861597,2578
5,정상-only + 단건 계좌 거래 제거,500159,6577987,6576047,1940,0.029492,3389.71,346054,344429,1625
6,정상-only + 1홉 계좌 전체 제거,484828,5773823,5772836,987,0.017094,5848.87,1150218,1147640,2578


## 8. 자금세탁 넘기는 비율 분석

In [78]:
import numpy as np
import pandas as pd


def calculate_pass_through(
    trans,
    anchor_label=1,
    horizons=(1, 24, 168),
    max_anchors=None,
):
    """
    anchor_label=1:
        자금세탁 거래를 수령한 계좌가 이후 얼마를 송금했는지 계산

    horizons:
        수령 후 누적 송금을 계산할 시간 단위
        1, 24, 168시간 = 1시간, 1일, 7일
    """

    df = trans.reset_index(drop=True).copy()
    df["_row_id"] = np.arange(len(df))

    df["Timestamp"] = pd.to_datetime(df["Timestamp"])

    # 은행과 계좌번호를 결합한 계좌 복합키
    df["_from_account"] = (
        df["From Bank"].astype("string")
        + "::"
        + df["Account"].astype("string")
    )
    df["_to_account"] = (
        df["To Bank"].astype("string")
        + "::"
        + df["Account.1"].astype("string")
    )

    # 분석 기준이 되는 수령 거래
    anchors = df[
        df["Is Laundering"].eq(anchor_label)
        & df["Amount Received"].gt(0)
        & df["Timestamp"].notna()
    ].copy()

    if max_anchors is not None and len(anchors) > max_anchors:
        anchors = anchors.sample(
            n=max_anchors,
            random_state=42,
        )

    # 동일 계좌·동일 통화의 후속 송금만 찾기 위한 키
    anchors["_flow_key"] = (
        anchors["_to_account"]
        + "||"
        + anchors["Receiving Currency"].astype("string")
    )

    needed_keys = set(anchors["_flow_key"])

    # 자기거래는 '다음 계좌로 보냄'에 해당하지 않으므로 제외
    outgoing = df[
        df["_from_account"].ne(df["_to_account"])
        & df["Amount Paid"].gt(0)
        & df["Timestamp"].notna()
    ].copy()

    outgoing["_flow_key"] = (
        outgoing["_from_account"]
        + "||"
        + outgoing["Payment Currency"].astype("string")
    )

    # 자금세탁 수령 계좌와 관련된 송금만 남겨 메모리 절약
    outgoing = outgoing[
        outgoing["_flow_key"].isin(needed_keys)
    ]

    # 계좌·통화별 후속 송금 배열 생성
    outgoing_groups = {}

    for flow_key, group in outgoing.groupby(
        "_flow_key",
        sort=False,
    ):
        group = group.sort_values("Timestamp")

        outgoing_groups[flow_key] = {
            "time": group["Timestamp"]
                .astype("int64")
                .to_numpy(),
            "amount": group["Amount Paid"]
                .astype(float)
                .to_numpy(),
            "label": group["Is Laundering"]
                .astype(int)
                .to_numpy(),
            "recipient": group["_to_account"]
                .astype(str)
                .to_numpy(),
            "timestamp": group["Timestamp"].to_numpy(),
        }

    records = []

    for _, anchor in anchors.iterrows():
        received = float(anchor["Amount Received"])
        anchor_time = pd.Timestamp(anchor["Timestamp"])
        anchor_time_ns = anchor_time.value

        record = {
            "anchor_row_id": anchor["_row_id"],
            "anchor_time": anchor_time,
            "account": anchor["_to_account"],
            "received_amount": received,
            "currency": anchor["Receiving Currency"],
            "anchor_label": anchor_label,
        }

        group = outgoing_groups.get(anchor["_flow_key"])

        if group is None:
            record["next_out_minutes"] = np.nan
            record["next_out_ratio_percent"] = np.nan

            for hour in horizons:
                record[f"all_out_ratio_{hour}h_percent"] = 0.0
                record[f"aml_out_ratio_{hour}h_percent"] = 0.0
                record[f"recipient_count_{hour}h"] = 0

            records.append(record)
            continue

        times = group["time"]
        amounts = group["amount"]
        labels = group["label"]
        recipients = group["recipient"]

        # 같은 분에 발생한 송금도 후보에 포함
        start = np.searchsorted(
            times,
            anchor_time_ns,
            side="left",
        )

        cumulative_all = np.concatenate(
            [[0.0], np.cumsum(amounts)]
        )
        cumulative_aml = np.concatenate(
            [[0.0], np.cumsum(amounts * labels)]
        )

        # 바로 다음 송금
        if start < len(times):
            next_amount = amounts[start]
            next_time_ns = times[start]

            record["next_out_minutes"] = (
                next_time_ns - anchor_time_ns
            ) / (60 * 1_000_000_000)

            record["next_out_ratio_percent"] = (
                next_amount / received * 100
            )

            record["next_recipient"] = recipients[start]
        else:
            record["next_out_minutes"] = np.nan
            record["next_out_ratio_percent"] = np.nan
            record["next_recipient"] = pd.NA

        # 기간 내 누적 송금
        for hour in horizons:
            end_time_ns = (
                anchor_time + pd.Timedelta(hours=hour)
            ).value

            end = np.searchsorted(
                times,
                end_time_ns,
                side="right",
            )

            all_out = (
                cumulative_all[end]
                - cumulative_all[start]
            )
            aml_out = (
                cumulative_aml[end]
                - cumulative_aml[start]
            )

            record[f"all_out_ratio_{hour}h_percent"] = (
                all_out / received * 100
            )
            record[f"aml_out_ratio_{hour}h_percent"] = (
                aml_out / received * 100
            )
            record[f"recipient_count_{hour}h"] = len(
                set(recipients[start:end])
            )

        records.append(record)

    return pd.DataFrame(records)

In [80]:
hi_flow = calculate_pass_through(
    processed_samples["HI"]["transactions"],
    anchor_label=1,
)

li_flow = calculate_pass_through(
    processed_samples["LI"]["transactions"],
    anchor_label=1,
)

In [81]:
flow_columns = [
    "next_out_minutes",
    "next_out_ratio_percent",
    "all_out_ratio_1h_percent",
    "all_out_ratio_24h_percent",
    "all_out_ratio_168h_percent",
    "aml_out_ratio_24h_percent",
    "recipient_count_24h",
]

hi_flow[flow_columns].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
).T

,count,mean,std,min,25%,50%,75%,90%,95%,max
next_out_minutes,2468.0,1877.711102,2373.304280,0.000000,308.000000,874.000000,2502.500000,5196.300000,7301.900000,1.466700e+04
next_out_ratio_percent,2468.0,12055.864156,228636.412951,0.000002,8.626208,52.071847,176.292724,1169.704519,4750.715775,1.022358e+07
all_out_ratio_1h_percent,5177.0,448.891701,25347.010247,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.805418e+06
all_out_ratio_24h_percent,5177.0,14556.938198,337354.697172,0.000000,0.000000,0.000000,12.541866,363.081930,3125.442054,1.964058e+07
all_out_ratio_168h_percent,5177.0,48765.356030,696501.943920,0.000000,0.000000,0.000000,243.879501,3849.173375,30234.187072,3.347407e+07
aml_out_ratio_24h_percent,5177.0,1233.862492,48199.024681,0.000000,0.000000,0.000000,0.000000,0.000000,95.091888,3.240953e+06
recipient_count_24h,5177.0,0.530423,1.112964,0.000000,0.000000,0.000000,1.000000,2.000000,3.000000,1.100000e+01


In [82]:
summary = pd.Series({
    "1시간 이내 송금 비율": (
        hi_flow["next_out_minutes"].le(60)
    ).mean() * 100,

    "1시간 내 80% 이상 송금": (
        hi_flow["all_out_ratio_1h_percent"].ge(80)
    ).mean() * 100,

    "24시간 내 80% 이상 송금": (
        hi_flow["all_out_ratio_24h_percent"].ge(80)
    ).mean() * 100,

    "24시간 내 2개 이상 계좌로 송금": (
        hi_flow["recipient_count_24h"].ge(2)
    ).mean() * 100,
})

summary

,0
1시간 이내 송금 비율,3.090593
1시간 내 80% 이상 송금,1.506664
24시간 내 80% 이상 송금,18.891250
24시간 내 2개 이상 계좌로 송금,11.493143


#### 패턴별 비교

In [93]:
hi_trans = processed_samples["HI"]["transactions"]
li_trans = processed_samples["LI"]["transactions"]

hi_patterns = pattern_transactions["HI"]
li_patterns = pattern_transactions["LI"]

# 여러 번 실행해도 기존 라벨 컬럼 충돌이 없도록 제거 후 연결
hi_flow_labeled = attach_pattern_labels(
    flow=hi_flow.drop(
        columns=["Pattern Type", "Attempt ID"],
        errors="ignore",
    ),
    trans=hi_trans,
    patterns=hi_patterns,
)

li_flow_labeled = attach_pattern_labels(
    flow=li_flow.drop(
        columns=["Pattern Type", "Attempt ID"],
        errors="ignore",
    ),
    trans=li_trans,
    patterns=li_patterns,
)

In [94]:
print(
    "HI 전체:",
    len(hi_flow_labeled),
    "패턴 연결:",
    hi_flow_labeled["Attempt ID"].notna().sum(),
    "패턴 외:",
    hi_flow_labeled["Attempt ID"].isna().sum(),
)

print(
    "LI 전체:",
    len(li_flow_labeled),
    "패턴 연결:",
    li_flow_labeled["Attempt ID"].notna().sum(),
    "패턴 외:",
    li_flow_labeled["Attempt ID"].isna().sum(),
)

HI 전체: 5177 패턴 연결: 3209 패턴 외: 1968
LI 전체: 3565 패턴 연결: 1023 패턴 외: 2542


In [95]:
def summarize_pass_through_by_pattern(flow):
    return (
        flow
        .groupby("Pattern Type", dropna=False)
        .agg(
            거래수=("anchor_row_id", "size"),

            다음송금_존재건수=(
                "next_out_minutes",
                "count",
            ),

            다음송금_중앙시간=(
                "next_out_minutes",
                "median",
            ),

            다음송금_중앙비율=(
                "next_out_ratio_percent",
                "median",
            ),

            일일_누적통과율=(
                "all_out_ratio_24h_percent",
                "median",
            ),

            일일_AML통과율=(
                "aml_out_ratio_24h_percent",
                "median",
            ),

            일일_수취인수=(
                "recipient_count_24h",
                "median",
            ),
        )
        .sort_values(
            "일일_누적통과율",
            ascending=False,
        )
    )


hi_pass_through_pattern_summary = (
    summarize_pass_through_by_pattern(
        hi_flow_labeled
    )
)

li_pass_through_pattern_summary = (
    summarize_pass_through_by_pattern(
        li_flow_labeled
    )
)

display(hi_pass_through_pattern_summary)
display(li_pass_through_pattern_summary)

,거래수,다음송금_존재건수,다음송금_중앙시간,다음송금_중앙비율,일일_누적통과율,일일_AML통과율,일일_수취인수
Pattern Type,,,,,,,
BIPARTITE,263,155,723.0,25.263567,0.0,0.0,0.0
CYCLE,287,173,471.0,90.976664,0.0,0.0,0.0
FAN-IN,318,153,1401.0,30.329636,0.0,0.0,0.0
FAN-OUT,342,142,745.5,22.615523,0.0,0.0,0.0
GATHER-SCATTER,716,406,1993.0,83.255281,0.0,0.0,0.0
RANDOM,191,100,452.5,91.514715,0.0,0.0,0.0
SCATTER-GATHER,626,301,849.0,20.420379,0.0,0.0,0.0
STACK,466,263,582.0,30.316250,0.0,0.0,0.0
패턴 외,1968,775,938.0,91.540430,0.0,0.0,0.0


,거래수,다음송금_존재건수,다음송금_중앙시간,다음송금_중앙비율,일일_누적통과율,일일_AML통과율,일일_수취인수
Pattern Type,,,,,,,
RANDOM,77,62,346.5,50.793856,96.806229,0.0,1.0
CYCLE,83,50,402.5,20.215171,13.532544,0.0,1.0
SCATTER-GATHER,182,135,491.0,15.029111,6.383075,0.0,1.0
GATHER-SCATTER,150,99,510.0,21.171521,0.368777,0.0,1.0
BIPARTITE,129,69,474.0,15.528093,0.000000,0.0,0.0
FAN-OUT,149,84,447.5,18.428972,0.000000,0.0,0.0
FAN-IN,73,43,1045.0,29.005772,0.000000,0.0,0.0
STACK,180,125,664.0,17.586739,0.000000,0.0,0.0
패턴 외,2542,1013,901.0,72.690696,0.000000,0.0,0.0
